# Ray 2.60：LLM 后训练实战入门

本 Notebook 用 Ray Train 在多 GPU 上做监督微调（SFT），重点讲清楚 Ray 如何负责分布式调度、数据分片、故障恢复和 checkpoint。后续章节说明如何扩展到 LoRA/QLoRA、DPO 与 GRPO。

**适用环境：** Linux + NVIDIA GPU（建议单机 1 张 24 GB 或更多显存的 GPU 起步；多卡需要 NCCL）。示例模型为 `Qwen/Qwen2.5-0.5B-Instruct`，实际模型需检查 Hugging Face 访问权限、许可证和显存。Windows 原生环境不建议直接跑多 GPU 训练，请使用 Linux 或 WSL2。

> 文档核对日期：2026-11-17。Ray 官方文档与 PyPI 当前版本为 2.60.0。Ray 版本迭代较快，升级版本时请回看官方 API。

## 1. 后训练与 Ray 组件

SFT 使用问答/指令样本优化 next-token loss。DPO 使用偏好对直接优化策略；GRPO/PPO 则需要 rollout、奖励模型或规则奖励。Ray 的角色是把工作拆成可调度的分布式任务：Ray Data 负责数据流水线，Ray Train 负责多 worker 训练，Ray Tune 可做超参搜索；在线推理可以另接 Ray Serve。

本教程先完成可检查的 SFT 工作流。真正的大规模偏好优化通常会用专门的训练框架（例如 verl、TRL 或 OpenRLHF），并由 Ray 集群承载其数据/训练/rollout 作业。

In [ ]:
# 先按 https://pytorch.org/get-started/locally/ 安装匹配 CUDA 的 PyTorch。
# 下面是教程选定的依赖组合，尚未在本机完成 GPU 集成验证；不要盲目升级所有包。
%pip install "ray[train,data,default]==2.60.0" "transformers==4.57.1" "datasets==4.2.0" "accelerate==1.10.1" "peft==0.17.1" safetensors
# 安装后重启 kernel；成功训练后使用 pip freeze 保存实际环境。

In [ ]:
import os
import ray
import torch

print("Ray:", ray.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. 启动 Ray 与集群检查

本地学习可以 `ray.init()`；生产多机环境应连接已经启动的 Ray cluster（head/worker 节点版本、Python 环境和依赖必须一致）。训练 worker 所需的软件可以用 runtime environment 安装，但大模型集群通常会预先构建一致的容器镜像。

In [ ]:
if not ray.is_initialized():
    ray.init()  # 已有集群时改为 ray.init(address="auto")

print(ray.cluster_resources())
print(ray.available_resources())

## 2.1 先用 CPU 理解 Ray：Task、ObjectRef、Actor

Task 适合独立计算，例如每个回答的规则奖励；Actor 是持有状态的进程，例如常驻 GPU 的 rollout 模型。
`.remote()` 立即返回 ObjectRef，`ray.get()` 等待结果。不要在提交循环里逐个 get，否则会串行等待。
下面仅计算玩具规则奖励，没有调用 LLM，也不是完整 RL 训练。

In [ ]:
@ray.remote(num_cpus=1)
def rule_reward(answer, expected):
    return float(answer.strip() == expected)

refs = [rule_reward.remote(answer, "4") for answer in ["4", "5", " 4 "]]
print("parallel rewards:", ray.get(refs))

@ray.remote(num_cpus=1)
class RewardCounter:
    def __init__(self):
        self.count = 0
    def record(self, rewards):
        self.count += len(rewards)
        return {"samples": self.count, "mean_reward": sum(rewards) / len(rewards)}

counter = RewardCounter.remote()
print(ray.get(counter.record.remote(ray.get(refs))))
ray.kill(counter)  # 释放 Actor 的 CPU 资源

**资源不是内存配额。** `num_gpus=1` 是调度声明，不能自动限制显存；fractional GPU 也不会隔离显存。
GPU Actor 的模型应在 `__init__` 加载一次，推理多次复用。训练和 rollout 同时占满 GPU 会导致等待甚至资源死锁。
大规模任务使用 `ray.wait()` 限制在途 ObjectRef 数量，避免把所有生成结果同时堆进 object store。

## 3. 构造一份小型指令数据

先用少量合成样本验证代码链路，不能用它衡量模型效果。生产训练时把数据换成经过授权、去重、清洗和脱敏的语料；至少做训练/验证切分，并检查格式、长度分布、重复率和隐私信息。聊天模板必须与目标模型匹配。

In [ ]:
from datasets import Dataset

# 按独立问题划分，再重复训练样本，避免同一个问题泄漏进验证集。
examples = [
    {"instruction": "用一句话解释 Ray。", "response": "Ray 是用于分布式 Python 计算的框架。"},
    {"instruction": "什么是 SFT？", "response": "SFT 使用带目标答案的数据继续训练模型。"},
    {"instruction": "Python 如何反转列表？", "response": "可以使用 items[::-1]。"},
    {"instruction": "checkpoint 有什么作用？", "response": "它保存模型及训练状态，用于评估或恢复。"},
]
train_ds = {
    "train": Dataset.from_list(examples[:3] * 32),
    "test": Dataset.from_list(examples[3:]),
}
train_ds

## 4. 使用 Ray Data 做可扩展预处理

Ray Data 适合将大规模文件读取、过滤、转换、分片等操作并行化。下面把 Hugging Face Dataset 转换为 Ray Dataset，并用模型 chat template 形成训练文本。简单示例中 tokenizer 在训练 worker 内调用；大规模训练应基于吞吐、CPU/GPU 利用率和缓存策略决定是否预先 tokenize。

In [ ]:
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
# 正式实验填模型 commit hash，避免 main 分支变化。
MODEL_REVISION = "main"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def to_chat_text(row):
    messages = [
        {"role": "user", "content": row["instruction"]},
        {"role": "assistant", "content": row["response"]},
    ]
    return {"text": tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )}

ray_train_ds = ray.data.from_huggingface(train_ds["train"]).map(to_chat_text)
ray_valid_ds = ray.data.from_huggingface(train_ds["test"]).map(to_chat_text)
ray_train_ds.take(2)

## 5. Ray Train + LoRA SFT

默认采用 LoRA，只训练注意力投影上的 adapter。设置 `USE_LORA=False` 才是全量微调。
Ray 启动每卡一个进程，Transformers/Accelerate 建立 DDP，Trainer 对每个 worker 中相同的 HF Dataset 做分布式采样。
**这里不能再手动 Ray 分片，否则会发生重复分片。**

为降低集成复杂度，小数据教程用 `take_all()` 收集处理结果；生产数据不要照搬。大数据改用 Ray Train `datasets` / `get_dataset_shard()` 配合自定义 DataLoader，或者让每个 worker 读取同一份磁盘 HF Dataset，由 Trainer 分布式采样。

当前 loss 覆盖完整聊天序列（含 user tokens），属于教学简化。真实指令 SFT 常只监督 assistant token；需要按模型模板生成 label mask，并逐条核对。不能仅按字符串位置猜测 token 边界。

有效全局 batch = 每卡 batch × 梯度累积步数 × worker 数。DDP 每卡仍持有完整基础模型；LoRA 不会分片模型权重。

In [ ]:
import tempfile
from pathlib import Path
from ray import train
from ray.train import Checkpoint, RunConfig, ScalingConfig
from ray.train.torch import TorchTrainer
from ray.train.huggingface.transformers import prepare_trainer

# 仅小数据演示允许收集到 driver。
train_rows = ray_train_ds.take_all()
eval_rows = ray_valid_ds.take_all()
MAX_LENGTH = 256
assert train_rows and eval_rows
print("train/eval rows:", len(train_rows), len(eval_rows))

In [ ]:
def train_loop_per_worker(config):
    import torch
    from datasets import Dataset
    from transformers import (
        AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments,
        DataCollatorForSeq2Seq, set_seed,
    )
    from peft import LoraConfig, get_peft_model

    set_seed(42)
    context = train.get_context()
    tok = AutoTokenizer.from_pretrained(config["model_id"], revision=config["revision"])
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "right"
    use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    model = AutoModelForCausalLM.from_pretrained(
        config["model_id"], revision=config["revision"],
        torch_dtype=torch.bfloat16 if use_bf16 else torch.float32,
    )
    model.config.use_cache = False
    if config["use_lora"]:
        model = get_peft_model(model, LoraConfig(
            task_type="CAUSAL_LM", r=8, lora_alpha=16, lora_dropout=0.05,
            target_modules=["q_proj", "v_proj"],
        ))
        model.print_trainable_parameters()

    def tokenize(batch):
        encoded = tok(batch["text"], truncation=True,
                      max_length=config["max_length"], add_special_tokens=False)
        encoded["labels"] = [ids.copy() for ids in encoded["input_ids"]]
        return encoded

    def make_dataset(rows):
        return Dataset.from_list(rows).map(tokenize, batched=True, remove_columns=["text"])

    # 只把 padding 的 labels 设为 -100；保留真实 EOS 的监督信号。
    collator = DataCollatorForSeq2Seq(tok, label_pad_token_id=-100, pad_to_multiple_of=8)
    with tempfile.TemporaryDirectory(prefix="ray-sft-") as workdir:
        args = TrainingArguments(
            output_dir=workdir, max_steps=config["max_steps"],
            per_device_train_batch_size=config["batch_size"],
            per_device_eval_batch_size=1,
            gradient_accumulation_steps=config["grad_accum"],
            learning_rate=config["learning_rate"],
            logging_steps=1, save_strategy="no", eval_strategy="no",
            bf16=use_bf16, fp16=False,
            gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False},
            ddp_find_unused_parameters=False, report_to=[], seed=42,
        )
        hf_trainer = Trainer(
            model=model, args=args, data_collator=collator,
            train_dataset=make_dataset(config["train_rows"]),
            eval_dataset=make_dataset(config["eval_rows"]),
        )
        hf_trainer = prepare_trainer(hf_trainer)
        output = hf_trainer.train()
        metrics = hf_trainer.evaluate()
        metrics["train_loss"] = output.training_loss
        checkpoint = None
        if context.get_world_rank() == 0:
            export = Path(workdir) / "export"
            hf_trainer.save_model(str(export))
            tok.save_pretrained(str(export))
            checkpoint = Checkpoint.from_directory(str(export))
        # 所有 rank 调用 report 相同次数；只由 rank 0 上传模型文件。
        train.report(metrics, checkpoint=checkpoint)

## 6. 启动训练

这是 10 个 optimizer steps 的 smoke run，目的在于检查工程链路，不能证明模型质量提高。
首次运行会下载模型。修改下方 `RUN_TRAINING=True` 才会实际启动。

默认 LoRA 学习率为 2e-4；全量微调需要另行选择（例如从 2e-5 开始试验）。
`storage_path` 本地目录只适合单机；多机必须换成所有节点可访问的持久存储。
Notebook 默认不自动开启重试，避免把 GPU OOM 反复重启成无效作业。

In [ ]:
RUN_TRAINING = False
NUM_WORKERS = 1
USE_LORA = True
result = None
if RUN_TRAINING:
    assert torch.cuda.is_available(), "请在具备 CUDA 的 Linux/WSL2 环境运行"
    assert ray.cluster_resources().get("GPU", 0) >= NUM_WORKERS
    trainer = TorchTrainer(
        train_loop_per_worker,
        train_loop_config={
            "model_id": MODEL_ID, "revision": MODEL_REVISION,
            "max_steps": 10, "max_length": MAX_LENGTH,
            "batch_size": 1, "grad_accum": 4,
            "learning_rate": 2e-4 if USE_LORA else 2e-5,
            "use_lora": USE_LORA,
            "train_rows": train_rows, "eval_rows": eval_rows,
        },
        scaling_config=ScalingConfig(
            num_workers=NUM_WORKERS, use_gpu=True,
            resources_per_worker={"CPU": 1, "GPU": 1},
        ),
        run_config=RunConfig(
            name="ray-llm-sft", storage_path=str(Path("ray_results").resolve()),
        ),
    )
    result = trainer.fit()
    print(result.metrics)
    print(result.checkpoint)
else:
    print("已跳过 GPU 训练；准备好环境后设置 RUN_TRAINING=True。")

## 7. 加载 adapter / 全量模型做生成检查

LoRA checkpoint 仅保存 adapter 和 tokenizer，推理时还必须加载原基础模型。下面在 CPU 上生成，速度较慢；有空闲 GPU 可把模型移到 CUDA。
验证集只有一个独立问题，不能作为可信 benchmark。正式评估需固定任务集、模型版本和解码参数，比较基础模型与微调模型的正确率、格式遵循和安全表现。

**恢复边界：** 本示例导出的是模型/adapter 权重，不含 optimizer、scheduler 和 RNG 状态，不能精确续训。精确恢复应让 HF Trainer 保存完整训练 checkpoint，并通过 Ray checkpoint 传递，再在 worker 内使用 `trainer.train(resume_from_checkpoint=...)`；各 rank 必须恢复同一训练步。

In [ ]:
if result is not None:
    from transformers import AutoModelForCausalLM
    from peft import PeftModel
    with result.checkpoint.as_directory() as checkpoint_dir:
        tok = AutoTokenizer.from_pretrained(checkpoint_dir)
        if USE_LORA:
            base = AutoModelForCausalLM.from_pretrained(
                MODEL_ID, revision=MODEL_REVISION, torch_dtype=torch.float32)
            model = PeftModel.from_pretrained(base, checkpoint_dir)
        else:
            model = AutoModelForCausalLM.from_pretrained(
                checkpoint_dir, torch_dtype=torch.float32)
        model.eval()
        messages = [{"role": "user", "content": "checkpoint 有什么作用？"}]
        prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tok(prompt, return_tensors="pt", add_special_tokens=False)
        with torch.inference_mode():
            out = model.generate(**inputs, max_new_tokens=64, do_sample=False,
                                 pad_token_id=tok.pad_token_id)
        print(tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
else:
    print("训练后再运行此单元。")

## 8. 从单机扩到集群

- 启动 Ray head 和 worker 节点，确认 `ray status` 能看到预期 GPU；Notebook 连接 `ray.init(address="auto")`。
- `ScalingConfig(num_workers=N, use_gpu=True)` 控制训练进程数量。训练方式（DDP、FSDP、DeepSpeed）决定显存切分和通信特性；多机大模型通常需要 FSDP/DeepSpeed，而非朴素 DDP。
- `RunConfig(storage_path=...)` 指向集群所有节点可访问的持久存储。验证中断恢复、checkpoint 完整性和依赖一致性。
- 用 Ray Data 读取 Parquet/JSONL 等分片文件，过滤坏样本、控制序列长度并观察数据吞吐；避免所有数据先读入 Notebook driver 内存。
- 训练前测量 tokens/s、GPU 利用率、数据等待时间和通信开销；加卡不一定线性提速。

## 9. LoRA、DPO 与 GRPO 的后续路径

**LoRA/QLoRA：** 在模型加载后用 PEFT 注入 adapter，并只训练 adapter 参数，可大幅降低优化器状态和梯度显存；QLoRA 还需要兼容的 4-bit bitsandbytes 环境。必须验证 Ray worker 的 CUDA、bitsandbytes 和 PEFT 版本组合。

**DPO：** 将数据转成 `(prompt, chosen, rejected)` 偏好对，使用 TRL `DPOTrainer` 或其它支持的训练循环。Ray Train 负责 worker 调度和资源管理；要特别关注 reference model 的显存开销、序列长度、偏好数据质量。

**GRPO/PPO：** 除策略训练外，还要调度 rollout 推理与奖励计算。多组件资源编排、异步采样和 checkpoint 语义更复杂，建议采用专用框架（如 verl）并按其官方 Ray 部署说明配置，而不是把本 Notebook 的 SFT 循环直接改名为 RL。

生产化还需要：确定性数据版本、可恢复的持久 checkpoint、实验追踪、离线评测套件、模型卡/许可证核查、内容安全评估，以及训练/推理格式兼容性测试。

## 官方资料

- [Ray Train 文档](https://docs.ray.io/en/latest/train/train.html)
- [Ray Train + Hugging Face Transformers](https://docs.ray.io/en/latest/train/api/doc/ray.train.huggingface.transformers.prepare_trainer.html)
- [Ray Data 文档](https://docs.ray.io/en/latest/data/data.html)
- [Ray 文档示例集](https://docs.ray.io/en/latest/ray-overview/examples.html)
- [Ray GitHub](https://github.com/ray-project/ray)

本 Notebook 展示 Ray 训练编排的基础模式。Ray 2.60.0 的 API、Transformers 的 Trainer 参数名，以及集成支持范围可能随版本变化；开跑前建议对照上述当前文档。

## 10. DPO 与 GRPO：从数据接口理解下一步

| 路线 | 数据 | 核心额外资源 | 优先检查 |
|---|---|---|---|
| SFT | prompt + answer | policy | assistant label mask、模板 |
| DPO | prompt + chosen + rejected | policy、reference | pair 质量、reference 与 policy 起点 |
| GRPO | prompt、可验证答案/奖励函数 | policy、rollout，按配置可有 reference | 组内奖励差异、KL、长度偏置 |

DPO 对同一 prompt 下 chosen/rejected 的相对 log-prob 优化；不需要在线 rollout。
GRPO 对同一 prompt 采样一组答案，根据相对奖励构造 advantage，再更新 policy。
若所有奖励都相同，组内标准化后没有有效相对信号；不能只看平均 reward，要看组内标准差、KL、输出长度与外部评测。

Ray 在这里负责进程和资源编排，不会自动提供 DPO/GRPO 的正确损失函数或权重同步。
TRL 适合从单机算法实验开始；verl/OpenRLHF 等有各自的 Ray 编排和版本要求。安装这些框架时使用其锁定环境，不能假设与此 Notebook 的依赖组合兼容。

In [ ]:
# 偏好数据的最小格式；不是训练调用。
preference_pair = {
    "prompt": [{"role": "user", "content": "2+2 等于多少？"}],
    "chosen": [{"role": "assistant", "content": "4"}],
    "rejected": [{"role": "assistant", "content": "5"}],
}
assert preference_pair["chosen"] != preference_pair["rejected"]

# GRPO 相对奖励的教学计算；实际训练还需 policy loss、KL、clip 等。
rewards = torch.tensor([1., 0., 1., 0.])
advantages = (rewards - rewards.mean()) / (rewards.std(unbiased=False) + 1e-6)
print("group advantages:", advantages.tolist())

## 11. 排查问题与练习

| 现象 | 检查与处理 |
|---|---|
| worker 一直 PENDING | `ray.available_resources()`；CPU/GPU 是否被 Actor 占满 |
| CUDA OOM | 降低长度/batch，LoRA；DDP 不会自动分片基础权重 |
| 多机 NCCL 卡住 | 网络互通、各节点 CUDA/驱动、rank 是否进入相同通信步骤 |
| loss 为 NaN | 输入标签、学习率、精度与 bf16 硬件支持 |
| checkpoint 无法续训 | 区分 adapter 导出与包含 optimizer/RNG 的完整训练状态 |
| 数据越多 driver 越慢 | 不要生产使用 `take_all()`；改为分片存储与流式读取 |

练习：① 改为两卡并计算有效 batch；② 固定全局 batch 比较吞吐；③ 为三条样本打印 token 与 label mask；④ 替换成独立验证集；⑤ 人为中断一个完整 checkpoint 训练并验证恢复步数。

当前 Notebook 未执行 GPU 训练，不包含伪造 loss、吞吐或效果指标。语法检查不能代替硬件环境下的集成测试。

### 补充官方链接

- [Ray 2.60.0 发布代码](https://github.com/ray-project/ray/tree/ray-2.60.0)
- [Ray Core 概念](https://docs.ray.io/en/latest/ray-core/key-concepts.html)
- [Ray Hugging Face 集成源码](https://github.com/ray-project/ray/blob/ray-2.60.0/python/ray/train/huggingface/transformers/_transformers_utils.py)
- [TRL DPO](https://huggingface.co/docs/trl/dpo_trainer)
- [TRL GRPO](https://huggingface.co/docs/trl/grpo_trainer)
- [verl](https://github.com/volcengine/verl)
- [OpenRLHF](https://github.com/OpenRLHF/OpenRLHF)

“最新”指核对日期的 Ray 发布版本；本教程未声称使用最新模型或所有依赖的最新版本。

## 12. vLLM 和 SGLang 在后训练中的位置

推理引擎虽然不负责计算梯度，但在后训练中同样重要：它们用于合成 SFT 数据、执行评估，以及为 GRPO/PPO 等算法生成在线 rollout。离线 DPO 使用固定偏好对，优化阶段不要求在线推理引擎。

| 层次 | 职责 | 示例 |
|---|---|---|
| 编排 | 调度进程、预留 CPU/GPU | Ray |
| 梯度计算 | 前向、反向传播与优化器更新 | Transformers、PyTorch、FSDP、DeepSpeed |
| 生成 | 批处理请求、管理 KV cache、采样输出 token | vLLM、SGLang |
| 算法集成 | 奖励、策略损失、rollout 收集与权重同步 | verl、OpenRLHF、TRL 的对应集成 |

vLLM 和 SGLang 都可以独立于 Ray 运行。引擎内部的张量并行，与 Ray 调度多个独立推理副本，是两种不同的机制。独立 HTTP 服务与 Ray 运行在同一台机器上，并不意味着该服务受 Ray 管理。

以下章节可选，无需先执行 SFT。它们使用基础模型，而不是前面导出的 LoRA adapter。加载 adapter 或合并后的 checkpoint，需要显式使用后端支持的流程。

## 13. 影响 Rollout 吞吐的推理概念

| 概念 | 含义 | 对后训练的影响 |
|---|---|---|
| Prefill（预填充） | 处理输入 token，建立注意力 KV 状态 | 长提示词可能主导首 token 延迟 |
| Decode（解码） | 利用缓存逐步生成输出 token | 长回答增加生成时间和 KV 显存占用 |
| KV cache | 为注意力保留各层 key/value 张量 | 权重放得下，不代表大批次也放得下 |
| Continuous batching（连续批处理） | 随生成进度动态加入新请求、移出已完成请求 | 使不同长度的请求更有效地共享 GPU |
| Prefix caching（前缀缓存） | 对相同 token 前缀复用符合条件的 KV 状态 | 重复系统提示可能减少 prefill；应分别测冷缓存和热缓存 |
| Tensor parallelism（张量并行，TP） | 将模型计算切分到多张 GPU | 能容纳更大的模型，但引入通信；并不等于多个独立副本 |

首 token 延迟（TTFT）衡量回答开始得多快；token 间延迟衡量解码响应速度；聚合输出 tokens/s 衡量多个请求的总体吞吐。最大吞吐配置可能有更差的尾延迟。

KV 显存大致随以下乘积增长：活跃序列 token 数 × 层数 × KV 头数 × 每头维度 × 两个张量（K、V）× 每元素字节数。具体还受后端分片和额外开销影响。只计算模型权重大小不足以估计服务容量。

两个引擎都支持高吞吐服务，并持续优化缓存和调度。vLLM 以分页 KV-cache 管理著称；SGLang 强调 RadixAttention 等机制带来的前缀复用。这些并非互斥的能力边界，应针对实际模型、后端版本、硬件和请求分布测量。

## 14. 在独立环境中启动一个推理后端

每个引擎使用**独立的 Linux 终端和软件环境**。不要将两个引擎直接装进前面固定版本的 SFT 环境：PyTorch、CUDA、计算内核和 Python 要求可能不同。按当前安装指南选择受支持的版本，记录 `pip freeze`、模型 revision、GPU、驱动和 CUDA 版本。下面是需要手动执行的终端命令，不会随 Notebook 自动执行。

### 方案 A：vLLM

核对时官方快速入门推荐使用 `uv` 自动选择 PyTorch 后端。先建立专用环境：

```bash
python -m venv .venv-vllm
source .venv-vllm/bin/activate
python -m pip install uv
uv pip install vllm --torch-backend=auto
vllm serve Qwen/Qwen2.5-0.5B-Instruct --host 127.0.0.1 --port 8000
```

### 方案 B：SGLang

建立另一个环境，按 [安装指南](https://docs.sglang.io/) 安装兼容的 SGLang 版本，然后运行：

```bash
python -m sglang.launch_server --model-path Qwen/Qwen2.5-0.5B-Instruct --host 127.0.0.1 --port 8001
```

单 GPU 上一次只启动一个服务。用 Ctrl+C 停止当前服务，并确认其 GPU 进程退出，再启动另一个。等待服务就绪后再请求。没有明确分配设备和显存时，不要同时运行这些服务和前面的训练任务。

对于较大模型，vLLM 提供 `--tensor-parallel-size`，SGLang 提供 `--tp-size`。先查看所安装版本的 `--help`，并预留对应数量的 GPU；设置 TP 不会凭空增加硬件。

示例绑定本地回环地址，并假设本地服务未开启认证。如果 Notebook 在另一台主机上，回环地址指向的是那台主机，需要改用可达地址或隧道。将服务开放到本地开发环境之外前，应配置认证。

In [ ]:
# 如有需要，仅在当前 Notebook kernel 中安装轻量客户端。
# %pip install openai

RUN_INFERENCE = False
BACKEND = "vllm"  # 启动 SGLang 服务后改为 "sglang"。
ENDPOINTS = {
    "vllm": "http://127.0.0.1:8000/v1",
    "sglang": "http://127.0.0.1:8001/v1",
}
INFERENCE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

def make_local_client(backend):
    import os
    from openai import OpenAI
    return OpenAI(
        base_url=ENDPOINTS[backend],
        api_key=os.environ.get("LOCAL_INFERENCE_API_KEY", "EMPTY"),
        timeout=120.0, max_retries=0,
    )

if RUN_INFERENCE:
    with make_local_client(BACKEND) as client:
        available_models = [m.id for m in client.models.list().data]
        print("服务提供的模型 ID：", available_models)
        assert INFERENCE_MODEL in available_models, "请使用服务实际提供的模型 ID。"
        response = client.chat.completions.create(
            model=INFERENCE_MODEL,
            messages=[{"role": "user", "content": "用两句话解释 KV 缓存。"}],
            temperature=0.0, top_p=1.0, max_tokens=96,
        )
        print(response.choices[0].message.content)
        print("Token 用量：", response.usage)
else:
    print("启动后端后，设置 RUN_INFERENCE=True 发送请求。")

### 14.1 使用 vLLM 离线生成

离线构造数据集时，可用 vLLM Python API，无需单独维护 HTTP 服务。下面的单元只能在兼容 vLLM 的 kernel 中运行，并确保有空闲 GPU 显存。它是服务模式的替代方案，不应在同一张已被占用的 GPU 上再启动一份引擎。

`LLM.chat()` 接收消息列表并应用模型聊天模板。使用 `LLM.generate()` 传入原始 prompt 时，应显式应用模板，避免遗漏或重复套用。调用结束后引擎仍可能占用 GPU；实验完成后重启专用 kernel 释放资源。

In [ ]:
RUN_OFFLINE_VLLM = False
if RUN_OFFLINE_VLLM:
    from vllm import LLM, SamplingParams
    engine = LLM(model=INFERENCE_MODEL, tensor_parallel_size=1)
    conversations = [
        [{"role": "user", "content": "什么是监督微调？"}],
        [{"role": "user", "content": "强化学习中的 rollout 是什么？"}],
    ]
    samples = engine.chat(
        conversations,
        sampling_params=SamplingParams(temperature=0.7, top_p=0.9, max_tokens=64),
    )
    for sample in samples:
        print(sample.outputs[0].text)
else:
    print("已关闭 vLLM 离线生成。")

## 15. 在相同负载下测量两个后端

分别为每个后端运行下一单元，修改 `BACKEND`，并在两次测量之间切换服务。保持 GPU 数量、模型 revision、dtype/量化方式、提示词集合、输出上限、并发数和缓存状态可比。显式传递采样参数可以减少隐藏差异，但后端默认行为和计算内核仍可能不同。

这个小型非流式测试报告端到端请求延迟和聚合输出 tokens/s，包含客户端及网络开销，不是严格的服务性能基准。它不测量 TTFT 或 token 间延迟，这些需要流式时间戳。重复 prompt 可能预热前缀缓存，应记录缓存条件。运行更快不代表回答更准确。

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from time import perf_counter

RUN_BENCHMARK = False
benchmark_results = globals().get("benchmark_results", {})
benchmark_prompts = [
    "用两句话解释梯度累积。",
    "用两句话解释张量并行。",
    "用两句话解释 KV 缓存。",
    "用两句话解释验证集。",
] * 2

def benchmark_request(prompt):
    with make_local_client(BACKEND) as client:
        start = perf_counter()
        response = client.chat.completions.create(
            model=INFERENCE_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0, top_p=1.0, max_tokens=64,
        )
        elapsed = perf_counter() - start
    if response.usage is None or response.usage.completion_tokens is None:
        raise ValueError("后端未返回输出 token 用量。")
    return elapsed, response.usage.completion_tokens

if RUN_BENCHMARK:
    start = perf_counter()
    with ThreadPoolExecutor(max_workers=2) as pool:
        measurements = list(pool.map(benchmark_request, benchmark_prompts))
    wall = perf_counter() - start
    durations = sorted(item[0] for item in measurements)
    benchmark_results[BACKEND] = {
        "requests": len(measurements), "concurrency": 2,
        "wall_seconds": wall,
        "mean_request_seconds": sum(durations) / len(durations),
        "max_request_seconds": max(durations),
        "output_tokens_per_second": sum(item[1] for item in measurements) / wall,
    }
    print(benchmark_results)
else:
    print("性能测试已关闭，尚无实测性能数据。")

## 16. 用 Ray 编排 Rollout 请求

下面的示例让 Ray Task 向已有的推理服务发送独立生成请求，再用精确匹配规则打分。这些 HTTP 客户端任务只申请 CPU。**它们不预留推理服务的 GPU。** 外部启动的服务独立于 Ray 占用 GPU。

这是单节点示例：worker 与 Notebook 使用同一个本地回环地址。集群部署需使用可达服务地址，并确保 worker 环境安装 `openai`。简单 HTTP 并发使用前面的线程池即可；当工作流还要分布式执行评分、预处理或其他计算时，Ray 更有价值。

若让 Ray 管理 GPU 引擎，应由长期存活的 Actor 预留 GPU，并只初始化一次引擎。TP 引擎需要协调预留全部所需 GPU。如果外层 Actor 已预留资源，而内层引擎又申请一组 Ray GPU worker，可能发生资源死锁。应采用集成框架支持的资源放置策略。

In [ ]:
RUN_RAY_ROLLOUT = False
rollout_records = []

if RUN_RAY_ROLLOUT:
    import os
    import ray
    if not ray.is_initialized():
        ray.init()

    @ray.remote(num_cpus=1, max_retries=0)
    def generate_rollout(base_url, model_id, prompt_id, sample_id, prompt):
        import os
        from openai import OpenAI
        with OpenAI(
            base_url=base_url,
            api_key=os.environ.get("LOCAL_INFERENCE_API_KEY", "EMPTY"),
            timeout=120.0, max_retries=0,
        ) as client:
            response = client.chat.completions.create(
                model=model_id,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.8, top_p=0.95, max_tokens=32,
            )
        answer = response.choices[0].message.content or ""
        return {
            "prompt_id": prompt_id, "sample_id": sample_id,
            "prompt": prompt, "answer": answer,
            "finish_reason": response.choices[0].finish_reason,
        }

    tasks = [("addition", i, "2+2 等于多少？只回复整数。") for i in range(4)]
    pending = []
    for prompt_id, sample_id, prompt in tasks:
        pending.append(generate_rollout.remote(
            ENDPOINTS[BACKEND], INFERENCE_MODEL, prompt_id, sample_id, prompt))
        if len(pending) >= 2:  # 限制在途请求数量。
            ready, pending = ray.wait(pending, num_returns=1)
            rollout_records.extend(ray.get(ready))
    rollout_records.extend(ray.get(pending))
    rollout_records.sort(key=lambda row: (row["prompt_id"], row["sample_id"]))
    for row in rollout_records:
        row["reward"] = float(row["answer"].strip() == "4")
    print(rollout_records)
else:
    print("已关闭 Ray rollout 请求。")

### 16.1 组内奖励只是 GRPO 的一部分

精确匹配奖励刻意保持简单：`4.` 或含 `4` 的解释都得零分。真实验证器需要仔细定义归一化与正确性。必须按 prompt 分组，不要跨无关问题归一化。下个单元使用总体标准差进行教学演示，实际框架可能采用不同的归一化约定。

这个单元和前面的 HTTP 示例都不计算策略梯度，没有执行策略更新或权重同步。

In [ ]:
from collections import defaultdict
import math

groups = defaultdict(list)
for row in rollout_records:
    groups[row["prompt_id"]].append(row)
for prompt_id, group in groups.items():
    assert len(group) >= 2, "GRPO 要求每个 prompt 有多个样本。"
    mean_reward = sum(row["reward"] for row in group) / len(group)
    std_reward = math.sqrt(sum((row["reward"] - mean_reward) ** 2 for row in group) / len(group))
    for row in group:
        row["illustrative_advantage"] = (row["reward"] - mean_reward) / (std_reward + 1e-6)
    print(prompt_id, {"mean_reward": mean_reward, "std_reward": std_reward})
    if std_reward == 0:
        print("该 prompt 的组内奖励没有相对差异信号。")

## 17. 从生成接口到正确的训练循环

同步训练流程如下：

```text
向 rollout worker 发布策略版本 k
    -> 生成标记为版本 k 的分组回答
    -> 计算奖励，准备 token 级训练数据
    -> 计算策略损失，更新可训练权重
    -> 将所有 rollout worker 同步到版本 k+1
    -> 按后端要求作废或刷新旧缓存
    -> 开始下一组 rollout
```

| 必需信息 | 原因 |
|---|---|
| 精确的输入/输出 token ID 与 mask | 将显示文本重新分词可能改变边界；需区分 padding、EOS 和截断 |
| 行为策略版本与 log probability | PPO/GRPO 变体的策略比率计算需要正确对齐的似然 |
| 采样配置 | temperature、top-p/top-k、重复惩罚和停止规则都会影响行为分布 |
| 参考策略输出（如算法需要） | 相对于指定参考策略计算 KL 惩罚 |
| 分组标识与奖励来源 | 将回答关联到 prompt、验证器版本和策略版本 |
| 已同步的权重或 adapter | 训练更新不会自动更新 HTTP 推理服务 |

部分 OpenAI-compatible 接口提供 `logprobs`，但其支持范围和语义因后端、模型、版本而异。返回的文本或 token 字符串不一定是可直接用于训练的精确 token-ID 序列。还要处理采样变换，以及训练端与推理端的数值差异。某些框架会重新计算行为策略 log-prob 或加入校正，必须遵守对应实现的数据约定。

只有服务实际确认加载某策略版本后，才能给 rollout 标记该版本。同步流程应等待所有推理副本更新完成再接受新任务；异步流程则必须跟踪并限制策略滞后，显式处理过时样本。

选择框架支持的权重传输路径：完整 checkpoint、adapter 加载与分布式内存传输是不同机制。检查同步是否同时处理 tokenizer/模板一致性，以及旧权重下生成的 KV/前缀缓存失效问题。Ray Train 导出 checkpoint 不会自动完成这些步骤。

### 17.1 GPU 放置与框架选择

| 布局 | 优势 | 主要约束 |
|---|---|---|
| 训练与 rollout 使用不同 GPU | 训练和生成可以重叠 | 需要更多设备/显存，并产生权重传输流量 |
| 共享 GPU，交替执行 | 复用有限 GPU | 显式释放、sleep/offload，或使用框架管理的内存切换 |
| 多个推理副本 | 提升独立 rollout 容量 | 请求路由、版本同步与队列上限 |
| 张量并行引擎 | 将大模型分摊到多张卡 | 协调资源预留与高带宽通信 |

单 GPU 先做顺序实验：停止训练 worker 后启动推理服务，停止服务后再恢复训练。GPU 份额不提供显存隔离。Ray placement group 可以协调资源预留，但不会自动解决训练与推理的显存兼容问题。

实际 GRPO/PPO 工作应选择 verl 或 OpenRLHF 已文档化的后端集成，并遵循其锁定环境。TRL 也为特定 Trainer 和模式提供 vLLM 集成。不能假设任意版本的 Ray、vLLM、SGLang 与训练框架都能互换；支持范围还取决于框架版本和模型架构。

选型需检查：模型/adapter 格式、token/log-prob 语义、权重传输、缓存失效、训练与生成同卡运行、版本追踪和 worker 故障恢复。生成 smoke test 通过只是第一步。

## 18. 推理练习、来源与验证边界

1. 两个后端使用相同 prompt，检查格式遵循与回答长度。
2. 在相同参数下测量单请求延迟和双请求并发，分别记录冷/热缓存条件。
3. 增加流式时间戳，测量 TTFT 与 token 间延迟，不要从总请求耗时推断。
4. 在用于策略损失之前，将后端的 token/log-prob 返回值与训练端分词结果核对。
5. 在支持的 RL 框架中更新一次策略，验证所有 rollout worker 都确认了新版本。

以下官方来源于 2026-11-17 核对，中文同步日期为 2026-11-18：

- [vLLM 快速入门](https://docs.vllm.ai/en/latest/getting_started/quickstart/)
- [vLLM 官方快速入门源码](https://github.com/vllm-project/vllm/blob/main/docs/getting_started/quickstart.md)：核对 `vllm serve`、`LLM`、`SamplingParams` 与聊天生成；当时渲染文档返回 HTTP 403，因此使用官方源码。
- [SGLang 请求教程](https://docs.sglang.io/docs/basic_usage/send_request)：核对 `sglang.launch_server` 与 OpenAI-compatible 聊天请求。
- [SGLang 文档与安装](https://docs.sglang.io/)
- [TRL vLLM 集成](https://huggingface.co/docs/trl/vllm_integration)
- [verl](https://github.com/volcengine/verl)
- [OpenRLHF](https://github.com/OpenRLHF/OpenRLHF)

这些链接指向持续更新的文档，并不是经过测试的版本锁。请记录实际安装版本，并遵循对应软硬件兼容矩阵。服务与 rollout 示例仅通过语法和 Notebook 格式检查；编写教程期间没有实际启动后端、下载模型、执行 GPU 性能测试、更新策略或进行分布式权重同步。

## 19. 从入门示例到独立实验：学习契约

对有研究训练的读者，目标是能解释“为什么有效、何时失效、如何验证”，而不仅是记住启动命令。原第 1–18 节是组件入门；第 19–27 节把正确性、实验设计和系统诊断串起来。

| 阶段 | 你必须交付什么 | 通过条件 |
|---|---|---|
| A：正确性 | 数据审计、token/label 表、微型优化实验 | 能定位泄漏、错误 mask、DPO 符号错误 |
| B：单卡学习 | 基础模型与 SFT 对比、独立验证集 | 小样本过拟合成功，真实任务指标可复核 |
| C：扩展与恢复 | 两卡对照、完整 checkpoint 恢复记录 | 全局 batch 对齐，无重复分片，恢复步数正确 |
| D：偏好优化 | DPO 或框架 GRPO 对照 | 固定基线，记录 KL/长度/奖励与外部正确率 |
| E：系统交付 | 性能报告、环境清单、失败复盘 | 可以在新环境复现，并解释主要瓶颈 |

先修自检：能推导交叉熵和 log-softmax 的梯度，解释 Adam 状态、DDP 梯度平均、采样分布与期望。不会的内容先通过小张量实验补齐。不要同时升级算法、模型、数据和后端，否则无法归因。

下面的 CPU 实验依赖仓库内 `labs/posttrain_checks.py`。请克隆完整仓库，并从仓库根目录打开 Notebook；它们不需要下载模型。

In [ ]:
from labs.posttrain_checks import (
    audit_splits, assistant_labels, dpo_loss, toy_dpo_update, paired_bootstrap, self_check,
)
print(self_check())

## 20. 数据契约与泄漏审计

先明确任务、输入分布、输出格式和判分规则，再收集数据。每条记录保留 ID、来源、授权范围、时间、任务类别、版本和清洗记录。按用户、文档、代码库或问题家族分组后切分；同一道题的改写不能简单随机分到训练与测试。

下面检查 schema、重复 ID 和归一化后的跨集 prompt 重复。它只能发现精确泄漏；语义重复、答案泄漏、基准污染和同来源近重复还需要额外审计。先切分再扩增，测试集保持封存，不用它选择超参。

训练前报告：各任务样本数、长度 P50/P95/P99、截断比例、零监督样本数、来源占比与重复率。若长回答更容易被截断，截断会改变任务分布。

In [ ]:
splits = {
    "train": [{"id": "train-1", "prompt": "What is 2+2?", "answer": "4"}],
    "test": [{"id": "test-1", "prompt": " what IS 2+2? ", "answer": "4"}],
}
issues = audit_splits(splits)
print(issues)
assert any(issue["kind"] == "cross_split_prompt" for issue in issues)
# Deliberately contaminated fixture: quarantine/re-split its source group before training.

## 21. SFT 的正确性比 loss 下降更重要

因果语言模型通常在内部将 logits 与 labels 错开一位：位置 t 的 logits 预测位置 t+1 的标签。若 Trainer/模型已 shift，就不要再手动 shift。`-100` 表示忽略该标签，不是词表中的 token。有效监督必须位于 shift 后仍保留的位置。

assistant-only SFT 应将 user、system 和 padding 的 labels 设为 `-100`，保留目标 assistant 内容及模板约定的结束符。真实 EOS 与 padding 即使用相同 ID，也必须根据位置和 attention mask 区分。

第 5 节当前使用 full-sequence loss：这是明确的教学简化。下个实验验证 mask 应用逻辑，但不会假装自动修好聊天模板。生产集成需要替换 worker 的 tokenize 函数：先取得模板可靠提供的 assistant mask，截断/填充时同步处理三个数组，再生成 labels。

某些 tokenizer 模板支持 `apply_chat_template(..., return_dict=True, return_assistant_tokens_mask=True)`，但模板必须具备 generation 标记。不要假设当前 Qwen 模板支持；不支持就使用经过验证的模板或整理可靠的 token 级边界，不能静默退回全序列监督。逐 token 打印角色、ID、文本、label；检查多轮、工具调用、EOS、超长截断与空回答。

Packing 也不是免费优化：拼接多个样本可能允许跨样本注意力。必须决定是否需要独立 attention 边界/位置重置，并确认内核支持。变长样本下，按 token 平均与按序列平均是不同目标；多卡和梯度累积的 loss 归一化也应核对。

In [ ]:
ids = [10, 11, 20, 2, 2]
attention = [1, 1, 1, 1, 0]
assistant = [0, 0, 1, 1, 0]
labels = assistant_labels(ids, attention, assistant)
print(list(zip(ids, attention, assistant, labels)))
print("Targets predicted by logits at positions 0..3:", labels[1:])
assert labels[-2:] == [2, -100]
# Synthetic token IDs test alignment; they do not represent a real tokenizer template.

## 22. 把 DPO 公式变成可以检查的更新

设 (x) 为 prompt，(y_w,y_l) 为偏好/非偏好回答，参考策略固定。序列 log-prob 只对回答 token 求和：

\[
L=-\log\sigma\left(\beta\left[\log\pi_\theta(y_w|x)-\log\pi_\theta(y_l|x)-\log\pi_{ref}(y_w|x)+\log\pi_{ref}(y_l|x)\right]\right).
\]

不要将序列概率直接连乘；使用 log-softmax 后求和。求和改成均值会改变目标，不能当成等价数值优化。参考模型应冻结且处于 eval 模式；若从 SFT 继续，记录 policy/reference 初始化关系。

下例只有两个可能答案，用单个 log-odds 参数表示策略，参考分布为均匀分布。它真实执行 DPO 梯度更新，并用有限差分核对梯度方向，但不是 LLM 微调。初始策略等于参考策略时 loss 应为 log(2)。固定 chosen 为胜者会提高其概率；交换 chosen/rejected 应反转方向。

迁移到真实 DPO：按任务构造高质量偏好对，核对两答案共享同一 prompt，检查 truncation、reference log-prob 和 chosen/rejected 的长度分布。奖励 margin 上升并不能证明任务正确率提高。

In [ ]:
import math
trace = toy_dpo_update()
print(trace[0], trace[-1])
assert math.isclose(dpo_loss(-2, -3, -2, -3), math.log(2))
assert trace[-1]["chosen_probability"] > trace[0]["chosen_probability"]
assert trace[-1]["loss"] < trace[0]["loss"]

## 23. 建立评估闭环，而不是追训练 loss

先固定独立 prompt 集、判分器版本、解码配置与任务切片，再评估 base → SFT → DPO/GRPO。保存每条 prompt 的原始输出和分数。规则可验证任务优先使用确定性验证器；开放回答评估需抽样人工核对 judge 偏置、顺序偏置与长度偏好。

主指标应来自目标任务：例如数学正确率、可执行代码测试通过率、结构化输出有效率；同时报告长度、拒答率、超时率和任务切片。运行代码验证需隔离执行环境。训练 reward 与独立评估必须分开，防止 reward hacking。

下例演示按同一 prompt 配对比较，并 bootstrap 估计分差区间。输入是合成分数，不是模型结果。真正实验应按独立问题或来源组重采样；同一 prompt 的多个输出并不独立，不能当成更多独立问题。小样本区间非常不稳定，不能据此宣称有效。

先确定采样预算再报告 pass@k；不要把不同 k 或不同生成次数混为一谈。训练随机性需多 seed 复核，且不要用最佳一次替代总体结果。

In [ ]:
base_scores = [0, 1, 0, 1, 0, 1, 0, 1]
candidate_scores = [1, 1, 0, 1, 1, 1, 0, 1]
print("Synthetic paired comparison:", paired_bootstrap(base_scores, candidate_scores))

## 24. 显存预算、分布式归一化与性能诊断

训练显存包含权重、梯度、优化器状态、激活、临时通信缓冲与内存碎片。经典混合精度 Adam 全量训练常用约 16 bytes/参数作静态状态粗估（具体实现可不同），还没有计算激活。LoRA 主要减少可训练参数相关状态，不消除基础权重和激活。

DDP 每卡复制模型和优化器状态；ZeRO-1/2/3 逐步切分优化器、梯度和参数；FSDP 在配置下分片参数/梯度/优化器状态，但会产生 all-gather 与 reduce-scatter。TP 分割层内计算；pipeline parallelism 分割层，需要处理流水线气泡。选方案要基于显存与互联，而不是卡数。

| 观测 | 先验证的假设 | 可做的对照 |
|---|---|---|
| GPU 常空闲、CPU 忙 | tokenize/读取成为瓶颈 | 缓存小批数据，比较 step 时间 |
| 多卡变慢 | 通信、过小 microbatch | 固定全局 batch，拆分计算/通信时间 |
| 显存充足但吞吐低 | 小 batch、padding 浪费 | 长度分桶，报告有效 token 吞吐 |
| rollout 尾部很慢 | 少数超长回答 | 看长度分位数、finish_reason 与截断率 |
| object store 增长 | 大量未消费 ObjectRef | 限制在途任务，及时消费结果 |

必须区分 microstep 与 optimizer step。全局样本 batch = microbatch × accumulation × world size；可变长度下每步的有效 token 数仍不同。比较一卡/两卡梯度时固定样本、seed、dropout 设置与归一化规则，允许浮点归约误差。不要只比较日志里一个平均 loss。

使用 PyTorch profiler/Nsight 等抓短窗口，预热后测量；GPU 操作异步，手动计时需正确同步。记录平均值和尾部延迟，加入 profiler 本身的开销说明。

## 25. 恢复、幂等性与实验清单

模型导出不等于训练恢复。完整恢复应记录：模型/adapter、optimizer、scheduler、AMP scaler（如使用）、随机数状态、global step、数据顺序/游标、梯度累积边界和框架配置。优先在 optimizer-step 边界保存，避免遗漏部分累积梯度。

验收：固定 seed 跑 N 步；另一路在 K 步保存并停止，从 checkpoint 恢复至 N；比较 step、学习率、数据 ID 顺序、optimizer 状态与参数差异。非确定性内核可能阻止 bitwise 相同，应事先定义容差并解释差异。不能仅以“成功加载”判断恢复正确。

Ray task 重试可能重复外部副作用；给输出使用稳定 sample ID 与原子提交/去重。Actor 重启后内存状态不会自动恢复；rollout worker 必须重新确认策略版本。重试不能修复持续 OOM 或错误数据。

下面只写本地实验清单，不填造硬件与指标；模型 revision、数据哈希、环境锁、代码 commit 和评估配置都应成为每次实验的固定输入。

In [ ]:
import hashlib
import importlib.metadata
import json
import platform
import subprocess
from pathlib import Path

def package_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

git_result = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True)
manifest = {
    "python": platform.python_version(),
    "git_commit": git_result.stdout.strip() if git_result.returncode == 0 else None,
    "packages": {p: package_version(p) for p in ["ray", "torch", "transformers", "peft", "vllm", "sglang"]},
    "model_revision": "REPLACE_WITH_COMMIT_HASH",
    "dataset_sha256": "REPLACE_WITH_REAL_DATA_HASH",
    "seed": 42, "hardware": "RECORD_ON_TARGET_HOST",
    "evaluation_config": "REPLACE_WITH_FROZEN_CONFIG",
    "status": "manifest_template_not_a_training_result",
}
artifact_dir = Path("artifacts")
artifact_dir.mkdir(exist_ok=True)
(artifact_dir / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(manifest)

## 26. GRPO 的失败模式与实验设计

组内标准化只是 advantage 构造的一种方式，不定义完整算法。实现还涉及 token/sequence 级目标、importance ratio、clip、KL、mask 与分母归一化。以框架指定版本的公式和源码为准，不要将不同变体的片段拼接。

| 现象 | 可能解释 | 区分方法 |
|---|---|---|
| reward 涨、独立正确率不涨 | reward hacking 或 judge 偏置 | 抽样复核、冻结外部验证器 |
| 所有组奖励相同 | 太容易/太难、采样不足或验证器坏了 | 看组内标准差，手工验证正负例 |
| 长度快速增长 | 长度偏置或停止规则问题 | 按长度切片看 reward/正确率/截断 |
| KL 激增 | 更新过大、旧样本或 reference 错配 | 检查版本、学习率、ratio 和 reference |
| reward 突然异常下降 | 权重未同步、tokenizer 不一致 | 检查每个副本加载确认与 token ID |

先做同步训练，证明一个 rollout→更新→同步循环正确，再考虑异步吞吐优化。实验表至少记录：policy/reference 版本、group size、采样参数、奖励版本、KL 系数、clip 范围、学习率、有效 batch、长度上限、策略滞后、失败/超时样本处理。

不要宣称 GRPO 在所有任务上优于 DPO：有可靠偏好对时先验证 DPO；有可验证奖励且在线探索有价值时再考虑 GRPO。

## 27. 毕业项目：交付一份能被复现的后训练实验

选择一个边界明确的任务，例如小型结构化抽取或可验证算术。以下是验收要求，不是已经完成的训练结果：

1. 写一页实验设计：业务/研究目标、主指标、数据来源、切分规则、预算与停止条件。
2. 建立 base 基线；在 8–32 条样本上过拟合，逐 token 核对监督，再换成有独立验证集的数据。
3. 单卡完成 SFT，保存完整 checkpoint；执行 K→N 恢复对照。若资源允许，再做两卡等全局 batch 对照。
4. 选 DPO 或框架支持的同步 GRPO，只改变一个主要因素；与 SFT 在同一封存评估集上比较。
5. 用 vLLM 或 SGLang 部署经过明确加载验证的权重/adapter，检查离线与服务输出差异，报告延迟、吞吐、长度及正确率。
6. 提交 manifest、配置、样本级评估、原始日志、失败案例和 1 页结论：效果是否可信，瓶颈在哪里，下一步如何证伪你的解释。

答辩问题：loss 降了为何正确率可能下降？两卡为何更慢？为什么同 ID 的 EOS 与 PAD 不能一起 mask？为什么导出 adapter 不等于精确恢复？rollout 的 log-prob 与训练端不一致时怎么办？如果回答仅是“换一个框架”，该项目还未通过。

新增 CPU 正确性检查已实际执行；本教程仍未完成 GPU SFT、真实 LLM DPO/GRPO、恢复或服务性能实测。它提供学习路径与可检查的组件，完整 GPU 项目需在你的硬件上按上述验收。

进一步阅读：[PyTorch DDP](https://docs.pytorch.org/docs/stable/generated/torch.nn.parallel.DistributedDataParallel.html)、[Transformers 聊天模板](https://huggingface.co/docs/transformers/chat_templating)、[TRL DPO](https://huggingface.co/docs/trl/dpo_trainer)、[TRL GRPO](https://huggingface.co/docs/trl/grpo_trainer)、[Ray 容错](https://docs.ray.io/en/latest/ray-core/fault_tolerance/index.html)。这些是后续阅读链接，不代表已经核对所有最新组合。

In [ ]:
# 完成本地学习后手动执行，释放本进程创建的 Ray runtime：
# ray.shutdown()